<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/16_ml_intro/16_4_2_Series_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas Series: Creating and Selecting Data

The previous notebook compared Series to Python lists and NumPy arrays conceptually. In this notebook we cover creating Series from lists, copying them, examining large ones, and selecting values by label, position, or condition.

---

## Learning Objectives

After working through this notebook, you should be able to:

- Create Series from lists, with and without a custom index
- Distinguish a copy from an alias, and know when each is created
- Select data by label (`.loc[]`), by position (`.iloc[]`), and by condition (boolean masks)
- Update values and index labels, and persist changes by reassignment

---

## Importing Pandas
By convention, Pandas is imported with the alias `pd`.

In [1]:
# imports
import pandas as pd

### Series and the Song Dataset

In the first notebook we worked with a song dataset that had columns for tempo, energy, and a danceability label. Here, we will store tempo and energy values for five songs as Pandas **Series**, and see that a Series can do everything the NumPy arrays from the previous notebook could do, plus more, once we give it a labeled index.

In [2]:
# Tempo and energy for five songs, each stored as a Pandas Series
# (each column of a Pandas table is a Series)
song_tempos = pd.Series([120, 95, 145, 130, 110], name='Tempo')
song_energies = pd.Series([0.8, 0.3, 0.9, 0.7, 0.5], name='Energy')
print('Tempos:')
print(song_tempos)
print()
print('Energies:')
print(song_energies)

# Series support the same stats methods we used on NumPy arrays
print()
print(f'Mean tempo: {song_tempos.mean()} BPM')
print(f'Max energy: {song_energies.max()}')

Tempos:
0    120
1     95
2    145
3    130
4    110
Name: Tempo, dtype: int64

Energies:
0    0.8
1    0.3
2    0.9
3    0.7
4    0.5
Name: Energy, dtype: float64

Mean tempo: 120.0 BPM
Max energy: 0.9


## Creating and Copying Series

### Creating Empty Series

To create a new series, we use `pd.Series()`. When we create an empty series, Pandas would like to know what kind of data we intend to store in it. We could specify integers with `dtype='int64'`, strings with `dtype='string'`, and [other things](https://pandas.pydata.org/pandas-docs/stable/user_guide/basics.html#basics-dtypes). If we want to mix data types within the series, we can specify 'object' as the data type using the `dtype='object'` argument.


In [3]:
# examples of creating empty series objects
my_integer_series = pd.Series(dtype='int64')
my_string_series = pd.Series(dtype='string')
my_mixed_series = pd.Series(dtype='object')

# check the empty series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')

Series([], dtype: int64), length = 0 
Series([], dtype: string), length = 0 
Series([], dtype: object), length = 0 


### Creating Series from Lists

More commonly, we won't start with empty series, instead we will create a series using existing data from some other kind of object. If we have the data in some form and want to put it into a series, we again use `pd.Series` but we do so using the data source as an argument. Below we create series from lists.

In [4]:
integer_list = [11, 12, 13, 14, 15]
string_list = ['apple', 'banana', 'cherry', 'daikon', 'eggplant']
mixed_list = integer_list + string_list

# examples of creating series objects from lists
my_integer_series = pd.Series(integer_list, dtype='int64')
my_string_series = pd.Series(string_list, dtype='string')
my_mixed_series = pd.Series(mixed_list, dtype='object')

# check the series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')
  print()


0    11
1    12
2    13
3    14
4    15
dtype: int64, length = 5 

0       apple
1      banana
2      cherry
3      daikon
4    eggplant
dtype: string, length = 5 

0          11
1          12
2          13
3          14
4          15
5       apple
6      banana
7      cherry
8      daikon
9    eggplant
dtype: object, length = 10 



When we create a series from existing data we do not have to specify the datatype using the `dtype` argument. If we exclude the argument, Pandas will attempt to infer the dtype of the series. (Notice in the output below that Pandas infers the dtype `str` for the list of fruit names, which is its default text type, rather than the `string` type we asked for explicitly above. For our purposes the two behave the same way.) Often this automatic inference works as you'd like, but there are occasions where the data you are building the series from cause some complications. We will return to this issue in a bit when we discuss dataframes.  

In [5]:
# examples of creating series objects from lists without specifying the data type
my_integer_series = pd.Series(integer_list)
my_string_series = pd.Series(string_list)
my_mixed_series = pd.Series(mixed_list)

# check the series objects
series_list = [my_integer_series, my_string_series, my_mixed_series]
for s in series_list:
  print(f'{s}, length = {len(s)} ')
  print()

0    11
1    12
2    13
3    14
4    15
dtype: int64, length = 5 

0       apple
1      banana
2      cherry
3      daikon
4    eggplant
dtype: str, length = 5 

0          11
1          12
2          13
3          14
4          15
5       apple
6      banana
7      cherry
8      daikon
9    eggplant
dtype: object, length = 10 



### Creating a Series with an Index
So far we've created series that have specified values, but no specified index. When we do this, Pandas assigns a range object to the index that generates integer values for each row, and we wind up with something that has an index much like a Python list.

We can specify an index using an optional argument when we construct the series. In the code below,  we are using two lists to specify the values and the index.

In [6]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

### Copying Series
Making a copy of a series works much like copying a list: assignment creates an alias (a second name for the same object), while `.copy()` creates a new independent object. We'll demonstrate this with series below.

When working with series, avoid making a copy unless you have a specific reason to do so. Copies consume extra memory and can slow your program down, especially with large datasets. Most of the time you can work with the original series or an alias.

In [7]:
# create a new series
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list, index=fruit_name_list)

# this does not make a copy of the series, only an alias
alias_not_a_new_series = my_fruit_series

# this makes a copy of the series
new_series_not_an_alias = my_fruit_series.copy()

In [8]:
# they contain the same values, note: ignore how this code works for now
print(alias_not_a_new_series.equals(new_series_not_an_alias))

# but they are not the same object
print(alias_not_a_new_series is new_series_not_an_alias)

# while these two are the same object
print(alias_not_a_new_series is my_fruit_series)

True
False
True


What we will usually do instead is simply *select* the part of the series we want to see. The selection operations covered in the Selecting Data section below (indexing, slicing, boolean masking) return a **new object containing the result and leave the original series unchanged**. If we want to keep a result, we assign it to a variable; otherwise it is displayed and discarded. The original series is never modified unless we explicitly assign the result back to it. This idea may feel a bit fuzzy for a while, but it will get better as we work through the exercises.

## Examining Series

We will often be working with series that contain a large number of values. If we have a series with 50000 values in it, and then we tell Python to print every value, we are going to have too much output to deal with. As a convenience, large series are usually displayed with values in the middle omitted.



In [9]:
# create a big list of random integers
import random
big_list = []
for v in range(50000):
  big_list.append(random.randint(1,100))

# convert the big list to a big series
big_series = pd.Series(data = big_list, dtype='int64')
len(big_series)

50000

In [10]:
# examine the big series; notice the truncation of the middle
big_series

0        52
1        60
2         9
3        99
4        99
         ..
49995    20
49996    48
49997    28
49998    45
49999    22
Length: 50000, dtype: int64

We can use the method `.head()` or `.tail()` to specifically look at the beginning or end of the series.

In [11]:
# using .head()
big_series.head()

0    52
1    60
2     9
3    99
4    99
dtype: int64

In [12]:
# using .tail()
big_series.tail()

49995    20
49996    48
49997    28
49998    45
49999    22
dtype: int64

By default, `.head()` and `.tail()` display five labels with their values. This can be modified using an optional argument.

In [13]:
# using .head() with an argument
big_series.head(10)

0    52
1    60
2     9
3    99
4    99
5    86
6    89
7    73
8    66
9    23
dtype: int64

In [14]:
# using .tail() with an argument
big_series.tail(3)

49997    28
49998    45
49999    22
dtype: int64

When working with data, we will often use `.head()` or `.tail()` to simply check that our code is working as intended, much like you would do with `print()` while writing your code.

## Selecting Data


### Selecting Data Using the Index
The lists we are used to working with have an index of integer values that start with 0 and increase by one for every value in the list. In contrast, a series has an index that can be anything we like. You can think of it as a second set of values that can be associated with the data in the series.

We will refer to the values in the index as labels. We can use the labels to retrieve subsets of data from our series.

In [15]:

# create lists of fruits and weights
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

# create series from lists
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

Notice that the datatype of the series is based on the values in the series, not on the values in the index.

Now that our series has both an index and values, we can access those values separately using dot notation.

In [16]:
# examine the series index attribute
my_fruit_series.index

Index(['apple', 'banana', 'cherry', 'dates', 'elderberry'], dtype='str')

In [17]:
# examine the series values attribute
my_fruit_series.values

array([180, 120,  15, 650, 450])

The values of a series index do not have to be unique.

In [18]:
# create lists for a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]

# constructs series with repeated values in the index
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple     180
apple     120
apple      15
banana    650
banana    450
dtype: int64

The values in an index also do not need to be sequential, and can be of any data type.

In [19]:
# create lists for constructing a series with a non-sequential index
fruit_name_list = ['banana', 'banana', 'apple', 'apple', 'apple']
fruit_weight_list = [180, 120, 15, 650, 450]

# constructs a series with a non-sequential index
my_fruit_series = pd.Series(data=fruit_name_list,
                            index=fruit_weight_list)
my_fruit_series

180    banana
120    banana
15      apple
650     apple
450     apple
dtype: str

### Selection by Label Using `.loc[]`
To get or update values in a series, we would use square brackets in much the same way we would with lists. However, we are going to add `.loc` before the brackets to specify that we are indexing or slicing based on the labels in the index.


In [20]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)
my_fruit_series

apple         180
banana        120
cherry         15
dates         650
elderberry    450
dtype: int64

In [21]:
# indexing a series values using .loc and an index label
my_fruit_series.loc['apple']

np.int64(180)

In [22]:
# indexing a series values using .loc and an index label
my_fruit_series.loc['dates']

np.int64(650)

We can also use label based slicing to get a range of values.

In [23]:
# note the label and value of cherry are included
my_fruit_series.loc['apple':'cherry']

apple     180
banana    120
cherry     15
dtype: int64

Note: Unlike normal Python slicing, which would usually go up to, but not include, the stop value, slicing with `.loc` includes the 'stop' value.

In [24]:
# slicing from a label until the end of the series
my_fruit_series.loc['cherry':]

cherry         15
dates         650
elderberry    450
dtype: int64

Since multiple values in the series can have the same 'label' or value for the index, indexing will return all values with a label, rather than just one.

In [25]:
# constructs a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]
my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

# indexing by index label when there are repeated values in index
my_fruit_series.loc['apple']

apple    180
apple    120
apple     15
dtype: int64

### Selection by Position Using `.iloc[]`

If you want to ignore the labels in the index and select values based on position, in the same way you do with a Python list, you can use 'implicit' indexing with `.iloc`.

Note: unlike `.loc`, slicing with `.iloc` works just like it does in python lists and strings; the stop value is not included.

In [26]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series.iloc[1:3]

banana    120
cherry     15
dtype: int64

We can't do negative indexing using `.loc`, since it's looking for labels, but we can do negative indexing with `.iloc`.

In [27]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series.iloc[-4:-1]

banana    120
cherry     15
dates     650
dtype: int64

We are going to spend a lot of time in this class asking Pandas to look at a series and identify some subset of values that are associated with a particular label or set of labels, so `.loc` will usually suffice. We will rarely use implicit indexing, so we are not going to practice it. However, it's important you know that it exists and that you recognize that when we use `.loc` we are relying on labels in the index rather than positions.



### Selection by Condition Using Booleans

When we use logical operators with a series, we get back a series full of Boolean values. Each value in the result is `True` if that element satisfied the condition, and `False` if it did not. The code cells below show a few examples.

In [28]:
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 45, 75]  # note: new, lighter weights than earlier examples

my_fruit_series = pd.Series(data=fruit_weight_list,
                            index=fruit_name_list)

my_fruit_series < 100

apple         False
banana        False
cherry         True
dates          True
elderberry     True
dtype: bool

In [29]:
my_fruit_series == 45

apple         False
banana        False
cherry        False
dates          True
elderberry    False
dtype: bool

In [30]:
my_fruit_series >= 100

apple          True
banana         True
cherry        False
dates         False
elderberry    False
dtype: bool

These series of booleans can be used as a mask to select specific values that meet a condition. We can do this by using the mask as an index.

In [31]:
# create the boolean mask and assign to a variable
heavy_fruit_mask = (my_fruit_series >= 100)

# use the mask variable to index into the series
my_fruit_series.loc[heavy_fruit_mask]

apple     180
banana    120
dtype: int64

We will be doing a lot of Boolean masking in this course. At some points, we will be combining three or four Boolean masks to select some particular subset of data. All of this can be done in a single line of code, but I am going to ask you to do it in multiple steps to help with troubleshooting your code.

In [32]:
# boolean masking done in a single step
my_fruit_series.loc[my_fruit_series <= 100]

cherry        15
dates         45
elderberry    75
dtype: int64

In [33]:
# the same thing done in two steps

# step 1: create the boolean mask and assign to a variable
light_fruit_mask = (my_fruit_series <= 100)

# step2: use the mask variable to index into the series
my_fruit_series.loc[light_fruit_mask]

cherry        15
dates         45
elderberry    75
dtype: int64

In [34]:
# an example using multiple boolean masks

# step 1: create the boolean mask and assign to a variable
light_fruit_mask = (my_fruit_series <= 20)

# step 2: create the boolean mask and assign to a variable
heavy_fruit_mask = (my_fruit_series >= 100)

# step 3: use the mask variables to index into the series
# the '|' substitutes for 'or'. Use | and & with masks (not 'and'/'or'),
# because Pandas evaluates element-wise, not the single True/False Python expects.
my_fruit_series.loc[light_fruit_mask | heavy_fruit_mask]

apple     180
banana    120
cherry     15
dtype: int64

## Updating Series

### Updating Values
Like a list, a series is mutable, so values can be updated using indexing with `.loc` or `.iloc`.

In [35]:
# create fruit weight series from lists
fruit_name_list = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
fruit_weight_list = [180, 120, 15, 45, 75]
my_fruit_series = pd.Series(data=fruit_weight_list, index=fruit_name_list)

# update a value based on a label
my_fruit_series.loc['cherry'] = 25
my_fruit_series

apple         180
banana        120
cherry         25
dates          45
elderberry     75
dtype: int64

In [36]:
# update a value based on a position
my_fruit_series.iloc[2] = 225
my_fruit_series

apple         180
banana        120
cherry        225
dates          45
elderberry     75
dtype: int64

You can also update multiple values simultaneously.

In [37]:
# update multiple values based on label
my_fruit_series.loc['apple':'cherry'] = [200, 150, 35]
my_fruit_series

apple         200
banana        150
cherry         35
dates          45
elderberry     75
dtype: int64

In [38]:
# update multiple values based on position
my_fruit_series.iloc[2:] = [300, 400, 500]
my_fruit_series

apple         200
banana        150
cherry        300
dates         400
elderberry    500
dtype: int64

Since multiple values can share the same label in the index, all the values that share a label can be updated with a single assignment.

In [39]:
# constructs a series with repeated values in index
fruit_name_list = ['apple', 'apple', 'apple', 'banana', 'banana']
fruit_weight_list = [180, 120, 15, 650, 450]

my_fruit_series = pd.Series(data = fruit_weight_list, index = fruit_name_list)
my_fruit_series

apple     180
apple     120
apple      15
banana    650
banana    450
dtype: int64

In [40]:
# updating multiple values with a single assignment
my_fruit_series.loc['apple'] = 333
my_fruit_series

apple     333
apple     333
apple     333
banana    650
banana    450
dtype: int64

Note: setting or 'updating' variables with `.loc` or `.iloc`, as we have in this section, changes the series directly. We do not need to make a copy of the series to make the change.

### Updating Index Labels
The series index object is not mutable, so we cannot use positional indexing to update a single label within the index. However, we can reassign a list of new values to the index.

In [41]:
my_fruit_series.index

Index(['apple', 'apple', 'apple', 'banana', 'banana'], dtype='str')

In [42]:
my_fruit_series.index = ['apple', 'banana', 'cherry', 'dates', 'elderberry']
my_fruit_series.index

Index(['apple', 'banana', 'cherry', 'dates', 'elderberry'], dtype='str')

---
Now that we can create, select, and update Series, the next notebook covers operating on them: vectorized math and string operations, sorting, and a couple of worked examples.

<a href="16_4_3_Series_Operations.ipynb">Continue to: Series Operations →</a>